# LungLens: train on Google Colab

This notebook runs the **same scripts** as the repository (`src/train.py`, `src/evaluate.py`, `scripts/sanity_check.py`), so results are reproducible and there is no hidden notebook-only code.

1. *Runtime > Change runtime type > T4 GPU* (CPU also works, just slower).
2. Add your Kaggle token as a Colab secret named `KAGGLE_API_TOKEN` (key icon in the left sidebar), or `KAGGLE_USERNAME` + `KAGGLE_KEY`.
3. Run all cells. At the end, download `lunglens.pt` and the `reports/` files and put them in the same folders of your local repo.

In [ ]:
# 1. Get the code. Edit REPO_URL if your GitHub repository has a different name.
REPO_URL = "https://github.com/RoboDevelopX/lunglens.git"
!git clone -q {REPO_URL} lunglens
%cd lunglens
!git log --oneline -3

In [ ]:
# 2. Install dependencies (Colab already has torch/torchvision with CUDA).
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
# 3. Kaggle credentials from Colab secrets -> environment variables (never printed).
import os
from google.colab import userdata
for name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        pass
print("credentials found:", [n for n in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY") if os.environ.get(n)])

In [ ]:
# 4. Download the dataset to local Colab disk (fast), outside the repo.
os.environ["LUNGLENS_DATA_DIR"] = "/content/data"
!python scripts/download_data.py --out /content/data

In [ ]:
# 5. Use the COMMITTED split (reports/splits.csv) so Colab trains on exactly the same images.
#    This just checks that every listed file exists; it does not re-split.
import pandas as pd
from pathlib import Path
splits = pd.read_csv("reports/splits.csv")
missing = [p for p in splits["path"] if not (Path("/content/data") / p).exists()]
print(splits["split"].value_counts().to_dict(), "| missing files:", len(missing))

In [ ]:
# 6. Train (two stages, early stopping on validation AUROC). Prints one line per epoch.
!python -m src.train

In [ ]:
# 7. Evaluate ONCE on the held-out test set and draw the figures.
!python -m src.evaluate

In [ ]:
# 8. Grad-CAM sanity checks: model randomisation test and attention on the image edges.
#    Writes reports/sanity.json and the sanity figures.
!python scripts/sanity_check.py

In [ ]:
# 9. Download the trained weights and the reports.
!zip -q -r lunglens_outputs.zip models/lunglens.pt reports
from google.colab import files
files.download("lunglens_outputs.zip")